In [1]:
import pandas as pd 
import numpy as np 

df_full = pd.read_csv("D1.csv")
df_clean = pd.DataFrame()
df_full.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,D1,18/08/2023,19:30,Werder Bremen,Bayern Munich,0,4,A,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D1,19/08/2023,14:30,Augsburg,M'gladbach,4,4,D,3,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D1,19/08/2023,14:30,Hoffenheim,Freiburg,1,2,A,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D1,19/08/2023,14:30,Leverkusen,RB Leipzig,3,2,H,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D1,19/08/2023,14:30,Stuttgart,Bochum,5,0,H,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df_full['Date'] = pd.to_datetime(df_full['Date'],format="%d/%m/%Y")
last_date_of_date = df_full['Date'].max()
df_full['daystoLast'] = (last_date_of_date-df_full['Date']).dt.days

In [3]:
team_map = {team:i for i,team in enumerate(sorted(df_full['HomeTeam'].unique()))}
outcome_map = {outcome:i for i,outcome in enumerate(sorted(df_full['FTR'].unique()))}
time_slot_map = {time_slot:i for i,time_slot in enumerate(sorted(df_full['Time'].unique()))}

In [5]:
def calculate_rolling_goal_avg(row, team_col, dataframe):
    team = row[team_col]
    current_date = row['Date']
    
    past_games = dataframe[
        ((dataframe['HomeTeam'] == team) | (dataframe['AwayTeam'] == team)) & 
        (dataframe['Date'] < current_date)
    ].sort_values('Date', ascending=False)
    
    goals = []
    for _, past_row in past_games.iterrows():
        if past_row['HomeTeam'] == team:
            goals.append(past_row['FTHG'])
        else:
            goals.append(past_row['FTAG'])
    
    last_5 = goals[:5]
    while len(last_5) < 5:
        last_5.append(0)
        
    return sum(last_5) / 5

def calculate_rolling_shot_avg(row, team_col, dataframe):
    team = row[team_col]
    current_date = row['Date']
    
    past_games = dataframe[
        ((dataframe['HomeTeam'] == team) | (dataframe['AwayTeam'] == team)) & 
        (dataframe['Date'] < current_date)
    ].sort_values('Date', ascending=False)
    
    goals = []
    for _, past_row in past_games.iterrows():
        if past_row['HomeTeam'] == team:
            goals.append(past_row['HST'])
        else:
            goals.append(past_row['AST'])
    
    last_5 = goals[:5]
    while len(last_5) < 5:
        last_5.append(0)
        
    return sum(last_5) / 5

def calculate_rolling_goal_avg(row, team_col, dataframe):
    team = row[team_col]
    current_date = row['Date']
    
    past_games = dataframe[
        ((dataframe['HomeTeam'] == team) | (dataframe['AwayTeam'] == team)) & 
        (dataframe['Date'] < current_date)
    ].sort_values('Date', ascending=False)
    
    goals = []
    for _, past_row in past_games.iterrows():
        if past_row['HomeTeam'] == team:
            goals.append(past_row['FTHG'])
        else:
            goals.append(past_row['FTAG'])
    
    last_5 = goals[:5]
    while len(last_5) < 5:
        last_5.append(0)
        
    return sum(last_5) / 5

def get_points(row, team_type):
    if row['FTR'] == 'D':
        return 1
    if team_type == 'Home':
        return 3 if row['FTR'] == 'H' else 0
    else:
        return 3 if row['FTR'] == 'A' else 0
    

In [6]:
def get_rolling_team_form(df, window=3):
    home_df = df[['Date', 'HomeTeam', 'HomePoints']].copy()
    home_df.columns = ['Date', 'Team', 'Points']
    home_df['Is_Home'] = True
    home_df['Original_Index'] = df.index

    away_df = df[['Date', 'AwayTeam', 'AwayPoints']].copy()
    away_df.columns = ['Date', 'Team', 'Points']
    away_df['Is_Home'] = False
    away_df['Original_Index'] = df.index

    all_matches = pd.concat([home_df, away_df]).sort_values(by=['Date', 'Original_Index'])

    all_matches['Form'] = all_matches.groupby('Team')['Points'].transform(
        lambda x: x.shift(1).rolling(window=window, min_periods=1).sum()
    )
    
    all_matches['Form'] = all_matches['Form'].fillna(0)

    home_form = all_matches[all_matches['Is_Home']].set_index('Original_Index')['Form']
    away_form = all_matches[~all_matches['Is_Home']].set_index('Original_Index')['Form']

    return home_form.reindex(df.index), away_form.reindex(df.index)

In [8]:
df_full['Home_Avg_Last5_Goal'] = df_full.apply(lambda row: calculate_rolling_goal_avg(row,'HomeTeam',df_full),axis=1)
df_full['Away_Avg_Last5_Goal'] = df_full.apply(lambda row: calculate_rolling_goal_avg(row,'AwayTeam',df_full),axis=1)
df_full['Home_Avg_Last5_Shot'] = df_full.apply(lambda row: calculate_rolling_shot_avg(row,'HomeTeam',df_full),axis=1)
df_full['Away_Avg_Last5_Shot'] = df_full.apply(lambda row: calculate_rolling_shot_avg(row,'AwayTeam',df_full),axis=1)
df_full['HomePoints'] = df_full.apply(lambda x: get_points(x, 'Home'), axis=1)
df_full['AwayPoints'] = df_full.apply(lambda x: get_points(x, 'Away'), axis=1)
df_full['Home_Last_3_Points'] = df_full.apply(lambda row:calculate_rolling_goal_avg(row,'HomeTeam',df_full),axis=1)
df_full['Away_Last_3_Points'] = df_full.apply(lambda row:calculate_rolling_goal_avg(row,'AwayTeam',df_full),axis=1)
df_full['Home_Form_Last_3'], df_full['Away_Form_Last_3'] = get_rolling_team_form(df_full, window=5)

In [9]:
#Date for Split
df_clean['date'] = df_full['Date']
#Data for Prediction
df_clean['timeSlot'] = df_full['Time'].map(time_slot_map)
df_clean['HomeTeam']=df_full['HomeTeam'].map(team_map)
df_clean['AwayTeam'] = df_full['AwayTeam'].map(team_map)
df_clean['homeGoal5GameAvgDiff'] = df_full['Home_Avg_Last5_Goal']-df_full['Away_Avg_Last5_Goal']
df_clean['homeShot5GameAvgDiff'] = df_full['Home_Avg_Last5_Shot'] - df_full['Away_Avg_Last5_Shot']
df_clean['homeTeamFormDiff'] = df_full['Home_Form_Last_3'] - df_full['Away_Form_Last_3']
df_clean['bookmakerHomeWinOdds'] = df_full['AvgH']
df_clean['bookmakerHomeDrawOdds'] = df_full['AvgD']
df_clean['bookmakerAwayWinOdds'] = df_full['AvgA']


#Predict Value
df_clean['homeWin'] = df_full['FTR'].map(outcome_map)

df_clean.head()

,date,timeSlot,HomeTeam,AwayTeam,homeGoal5GameAvgDiff,homeShot5GameAvgDiff,homeTeamFormDiff,bookmakerHomeWinOdds,bookmakerHomeDrawOdds,bookmakerAwayWinOdds,homeWin
0,2023-08-18,4,18,1,0.0,0.0,0.0,3.41,1.37,3.55,0
1,2023-08-19,0,0,12,0.0,0.0,0.0,2.39,1.66,2.43,1
2,2023-08-19,0,10,7,0.0,0.0,0.0,2.30,1.71,2.32,0
3,2023-08-19,0,11,14,0.0,0.0,0.0,2.24,1.77,2.28,2
4,2023-08-19,0,16,2,0.0,0.0,0.0,2.32,1.71,2.38,2


In [10]:
#Wheigt so that Data old Data is not as Relevant
df_clean['weight'] = np.exp(-df_full['daystoLast']/100)

In [11]:
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean = df_clean.sort_values('date').reset_index(drop=True)

In [12]:
features = ['timeSlot', 'homeGoal5GameAvgDiff','homeTeamFormDiff',
            'bookmakerHomeWinOdds', 'bookmakerHomeDrawOdds', 'bookmakerAwayWinOdds']
X = df_clean[features]
y = df_clean['homeWin']
weights = df_clean['weight']

In [ ]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
modell = xgb.XGBClassifier(
    n_estimators=1000,   
    max_depth=2,         
    learning_rate=0.05,   
    random_state=42,
    early_stopping_rounds=20, 
    enable_categorical=True,
    objective='multi:softprob', 
    eval_metric='mlogloss'
)

In [16]:
tscv = TimeSeriesSplit(n_splits=3)
scores_acc = []
scores_logloss = []

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    w_train, w_test = weights.iloc[train_index], weights.iloc[test_index]
    
    modell.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_train, y_train), (X_test, y_test)], 
        sample_weight_eval_set=[w_train, w_test],
        verbose=False 
    )
    
    acc = modell.score(X_test, y_test)
    scores_acc.append(acc)
    

    results = modell.evals_result()

    final_fold_logloss = results['validation_1']['mlogloss'][-1] 
    scores_logloss.append(final_fold_logloss)
    
    print(f"Fold Accuracy: {acc:.4f} | Fold Log Loss: {final_fold_logloss:.4f}")


Fold Accuracy: 0.3984 | Fold Log Loss: 1.0995
Fold Accuracy: 0.4375 | Fold Log Loss: 1.0650
Fold Accuracy: 0.5391 | Fold Log Loss: 1.0025


In [15]:

print("-" * 40)
print(f"Average Accuracy: {np.mean(scores_acc):.4f}")
print(f"Average Log Loss: {np.mean(scores_logloss):.4f}")

----------------------------------------
Average Accuracy: 0.4583
Average Log Loss: 1.0557
